In [1]:
import pandas as pd
import joblib

In [2]:
# ---- load data ----
dataset_type = "real"
df = pd.read_parquet(f'../data/cleaned/df_{dataset_type}_cleaned.parquet')

print(f'{dataset_type}: {df.shape}')

real: (3000, 38)


In [3]:
# ---- load pipeline ----
full_pipeline = joblib.load("../models/risk_pipeline.joblib")

In [4]:
# ---- features ----
cat_cols_direct = ['coverage_type', 'business_type', 'state']
cat_cols_with_unknown = ['payment_frequency']

numeric_cols = [
    'vehicle_count', 'num_heavy_vehicles', 'total_vehicle_count',
    'vehicle_avg_age', 'mileage_per_vehicle',
    'driver_count', 'driver_avg_age',
    'years_in_business',
    'prior_apd_claim_count', 'prior_al_claim_count', 'prior_loss_amount',
    'has_large_prior_loss', 'loss_per_claim',
    'late_payment_count',

    'deductible',
    'coverage_limit_000',
    'annual_premium',
    'risk_score_external', 'risk_score_external_isna',

    'policy_length_days', 'exposure_ratio',
]

In [5]:
# ---- create subset with feature columns ----
feature_cols = cat_cols_direct + cat_cols_with_unknown + numeric_cols

X2 = df[feature_cols]
X2.shape

(3000, 25)

In [6]:
# Generate probabilities
preds = full_pipeline.predict_proba(X2)[:, 1]

df["risk_score"] = preds
df[["policy_id", "risk_score"]].to_csv("../data/predictions/predictions.csv", index=False)
print("done -- predictions.csv written")

done -- predictions.csv written


In [7]:
df[["risk_score"]].describe()

,risk_score
count,3000.000000
mean,0.130536
std,0.090608
min,0.010392
25%,0.066157
50%,0.107966
75%,0.165560
max,0.771024


In [8]:
df = pd.read_csv("../data/predictions/predictions.csv")
print(df.shape)

df.head()

(3000, 2)


,policy_id,risk_score
0,POL-015001,0.206097
1,POL-015002,0.220538
2,POL-015003,0.230727
3,POL-015004,0.145544
4,POL-015005,0.192377
